<a href="https://colab.research.google.com/github/willy410-hub/generative-multi-agent-simulation/blob/main/Multi_Agent_Simulation_Groq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install termcolor > /dev/null
!pip install langchain langchain-community langchain-experimental langchain-classic
!pip install faiss-cpu
!pip install langchain-groq langchain-huggingface sentence-transformers

from datetime import datetime, timedelta
from typing import List
import math
import faiss
import os
import logging
logging.basicConfig(level=logging.ERROR)

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

from langchain_classic.retrievers import TimeWeightedVectorStoreRetriever

from termcolor import colored
from langchain_experimental.generative_agents import (
    GenerativeAgent,
    GenerativeAgentMemory,
)

In [ ]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [ ]:
USER_NAME = "Walid"  # The name you want to use when interviewing the agent.

LLM = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.7)

## Implementing Your First Generative Agent




In [ ]:


def relevance_score_fn(score: float) -> float:
    """Return a similarity score on a scale [0, 1]."""
    return 1.0 - score / math.sqrt(2)

def create_new_memory_retriever():
    """Create a new vector store retriever unique to the agent."""

    embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    embedding_size = 384
    index = faiss.IndexFlatL2(embedding_size)
    vectorstore = FAISS(
        embeddings_model.embed_query,
        index,
        InMemoryDocstore({}),
        {},
        relevance_score_fn=relevance_score_fn,
    )
    return TimeWeightedVectorStoreRetriever(
        vectorstore=vectorstore, other_score_keys=["importance"], k=15
    )

In [ ]:
Gem_memory = GenerativeAgentMemory(
    llm=LLM,
    memory_retriever=create_new_memory_retriever(),
    verbose=False,
    reflection_threshold=8,
)

# Defining the Generative Agent: Gem
Gem = GenerativeAgent(
    name="Gem",
    age=28,
    traits="curious, creative writer, world traveler",
    status="exploring the intersection of technology and storytelling",
    memory_retriever=create_new_memory_retriever(),
    llm=LLM,
    memory=Gem_memory,
)

In [ ]:
# The current "Summary" of a character can't be made because the agent hasn't made
# any observations yet.
print(Gem.get_summary())

In [ ]:
# We can add memories directly to the memory object

Gem_observations = [
    "Gem recalls her morning walk in the park",
    "Gem feels excited about the new book she started reading",
    "Gem remembers her conversation with a close friend",
    "Gem thinks about the painting she saw at the art gallery",
    "Gem is planning to learn a new recipe for dinner",
    "Gem is looking forward to her weekend trip",
    "Gem contemplates her goals for the month."
]

for observation in Gem_observations:
    Gem.memory.add_memory(observation)



# We will see how this summary updates after more observations to create a more rich description.
print(Gem.get_summary(force_refresh=True))

## Interacting and Providing Context to Generative Characters

## Pre-Interview with Character

Before sending our character on their way, let's ask them a few questions.

In [ ]:
def interview_agent(agent: GenerativeAgent, message: str) -> str:
    """Help the notebook user interact with the agent."""
    new_message = f"{USER_NAME} says {message}"
    return agent.generate_dialogue_response(new_message)[1]

In [ ]:
interview_agent(Gem, "What do you like to do?")


## Step through the day's observations.

In [ ]:
# Let's give Gem a series of observations to reflect on her day
# Adding observations to Gem' memory
Gem_observations_day = [
    "Gem starts her day with a refreshing yoga session.",
    "Gem spends time writing in her journal.",
    "Gem experiments with a new recipe she found online.",
    "Gem gets lost in her thoughts while gardening.",
    "Gem decides to call her grandmother for a heartfelt chat.",
    "Gem relaxes in the evening by playing her favorite piano pieces.",
]

for observation in Gem_observations_day:
    Gem.memory.add_memory(observation)


In [ ]:
# Let's observe how Gem's day influences her memory and character
for i, observation in enumerate(Gem_observations_day):
    _,reaction = Gem.generate_reaction(observation)
    print(colored(observation, "Red"), reaction)
    if ((i + 1) % len(Gem_observations_day)) == 0:
        print("*" * 40)
        print(
            colored(
                f"After these observations, Gem's summary is:\n{Gem.get_summary(force_refresh=True)}",
                "blue",
            )
        )
        print("*" * 40)


In [ ]:
def interview_agent(agent: GenerativeAgent, message: str) -> str:
    """Help the notebook user interact with the agent."""
    new_message = f"{USER_NAME} says {message}"
    return agent.generate_dialogue_response(new_message)[1]

In [ ]:
interview_agent(Gem, "What do you like to do?")

In [ ]:
# Let's give Gem a series of observations to reflect on her day
# Adding observations to Gem' memory
alexis_observations_day = [
    "Gem starts her day with a refreshing yoga session.",
    "Gem spends time writing in her journal.",
    "Gem experiments with a new recipe she found online.",
    "Gem gets lost in her thoughts while gardening.",
    "Gem decides to call her grandmother for a heartfelt chat.",
    "Gem relaxes in the evening by playing her favorite piano pieces.",
]

for observation in Gem_observations_day:
    Gem.memory.add_memory(observation)

In [ ]:
# Let's observe how Gem's day influences her memory and character
for i, observation in enumerate(Gem_observations_day):
    _, reaction = Gem.generate_reaction(observation)
    print(colored(observation, "Red"), reaction)
    if ((i + 1) % len(Gem_observations_day)) == 0:
        print("*" * 40)
        print(
            colored(
                f"After these observations, Gem's summary is:\n{Gem.get_summary(force_refresh=True)}",
                "blue",
            )
        )
        print("*" * 40)

# DialogueAgent and DialogueSimulator **Classes**
1. The DialogueAgent Class.

    
    Responsible for managing an individual agent's system prompt, message history, and LLM invocation.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq

class DialogueAgent:
    def __init__(
        self,
        name: str,
        system_message: SystemMessage,
        model: ChatGroq,
    ) -> None:
        self.name = name
        self.system_message = system_message
        self.model = model
        self.prefix = f"{self.name}: "
        self.reset()

    def reset(self):
        self.message_history = ["Here is the conversation so far."]

    def send(self) -> str:
        """
        Applies the chatmodel to the message history
        and returns the message string
        """
        message = self.model.invoke(
            [
                self.system_message,
                HumanMessage(content="\n".join(self.message_history + [self.prefix])),
            ]
        )
        return message.content

    def receive(self, name: str, message: str) -> None:
        """
        Concatenates {message} spoken by {name} into message history
        """
        self.message_history.append(f"{name}: {message}")

2. The DialogueSimulator Class.

    Orchestrates the conversation flow between multiple agents and control

In [ ]:
from typing import List, Callable

class DialogueSimulator:
    def __init__(
        self,
        agents: List[DialogueAgent],
        selection_function: Callable[[int, List[DialogueAgent]], int],
    ) -> None:
        self.agents = agents
        self._step = 0
        self.select_next_speaker = selection_function

    def reset(self):
        for agent in self.agents:
            agent.reset()

    def inject(self, name: str, message: str):
        """
        Initiates the conversation with a {message} from {name}
        """
        for agent in self.agents:
            agent.receive(name, message)

        # increment time
        self._step += 1

    def step(self) -> tuple[str, str]:
        # 1. choose the next speaker
        speaker_idx = self.select_next_speaker(self._step, self.agents)
        speaker = self.agents[speaker_idx]

        # 2. next speaker sends message
        message = speaker.send()

        # 3. everyone receives message
        for receiver in self.agents:
            receiver.receive(speaker.name, message)

        # 4. increment time
        self._step += 1

        return speaker.name, message